In [1]:
import json

import colorcet
import h5py
import matplotlib.pyplot as plt
import numpy as np

from lib.plots import set_axis

In [26]:
config_id = 68
#config_id = 101
#config_id = 307
filename = f"_outputs/output-{config_id}.h5"

with h5py.File(filename, "r") as store:
    phase_store = store["production"]
    config = json.loads(phase_store["metadata/config"][()])

    positions_history = []
    for step_key in phase_store[".steps"]:
        positions = phase_store[step_key]["positions"][:]
        positions_history.append(positions)
    positions_history = np.array(positions_history)

In [27]:
rg_history = np.sqrt(np.var(positions_history, axis=-2).sum(axis=-1))

In [ ]:
deltas = positions_history[:, None, :, :] - positions_history[:, :, None, :]
squared_dists = ((deltas - deltas[0]) ** 2).sum(axis=-1)

In [ ]:
subsample_size = None
random = np.random.Generator(np.random.PCG64(0))
rows, cols = np.triu_indices(squared_dists.shape[1], k=1)

if subsample_size:
    selection = random.choice(len(rows), size=subsample_size)
    rows = rows[selection]
    cols = cols[selection]

msds = squared_dists[:, rows, cols].reshape(squared_dists.shape[0], -1).mean(axis=1)
rmsds = np.sqrt(msds)

fig, ax = plt.subplots()
ax.plot(rg_history * 2, color="k", label=r"$ 2 R_\mathrm{g} $")
ax.plot(rmsds, color="r", label="pRMSD")
ax.legend(loc="lower right")
pass